# Lower Sorbian Dataset Analysis
This notebook creates the 3 needed datasets without lemma overlap.

In [1]:
# Step 1: Invert columns
input_file = 'dsb_original'
output_file = 'dsb'
with open(input_file, 'r', encoding='utf8') as fin, open(output_file, 'w', encoding='utf8') as fout:
    for line in fin:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            lemma, form, msd = parts
            fout.write(f'{lemma}\t{msd}\t{form}\n')
print(f'Inverted columns and saved to {output_file}')

Inverted columns and saved to dsb


In [6]:
# Step 2: Deterministic POS-stratified splits using all verbs and preserving global POS ratios
# Assumptions:
#  - We split ALL rows from input_file (no filtering), using POS from MSD prefix:
#      V = verbs, N = nouns, A = adjectives, O = other
#  - Target ratios are 80/10/10 for train/dev/test (edit the constants below as needed)
#  - Deterministic via seeded sampling per POS group (avoids blocky concatenation biases)

from collections import defaultdict
import os
import random
# Deterministic mixing seed to avoid overly-sorted outputs
random.seed(42)

input_file = "dsb"
# Configurable ratios
train_ratio, dev_ratio, test_ratio = 0.80, 0.10, 0.10

# Read all rows from the original source and invert to (lemma, msd, form)
all_by_pos = { 'V': [], 'N': [], 'A': [], 'O': [] }

with open(input_file, 'r', encoding='utf8') as fin:
    for raw in fin:
        parts = raw.rstrip('\n').split('\t')
        if len(parts) != 3:
            continue
        lemma, msd, form = parts
        # Normalize to standard order used by other datasets
        rec = (lemma, msd, form)
        # POS bucket by MSD prefix
        if msd.startswith('V'):
            all_by_pos['V'].append(rec)
        elif msd.startswith('N'):
            all_by_pos['N'].append(rec)
        elif msd.startswith('A'):
            all_by_pos['A'].append(rec)
        else:
            all_by_pos['O'].append(rec)

# Deterministic stable ordering within POS groups
for k in list(all_by_pos.keys()):
    # keep a deterministic sort so results are reproducible across runs
    all_by_pos[k] = sorted(all_by_pos[k], key=lambda t: (t[0], t[1], t[2]))
    # then mix locally for better variety
    random.shuffle(all_by_pos[k])

# Build splits preserving ratios per POS using seeded sampling
train_lines, dev_lines, test_lines = [], [], []

def split_group(group):
    n = len(group)
    if n == 0:
        return [], [], []
    # compute rounded allocation, then fix any rounding drift
    n_train = int(round(n * train_ratio))
    n_dev = int(round(n * dev_ratio))
    n_test = n - n_train - n_dev
    # fix negative or zero issues deterministically
    if n_test < 0:
        # reduce train first then dev until sums fit
        diff = -n_test
        while diff > 0 and n_train > 0:
            n_train -= 1
            diff -= 1
        while diff > 0 and n_dev > 0:
            n_dev -= 1
            diff -= 1
        n_test = n - n_train - n_dev
    # ensure small groups still give dev/test at least one sample when possible
    if n >= 3 and n_dev == 0:
        n_dev = 1
        if n_train > 0:
            n_train -= 1
        n_test = n - n_train - n_dev
    # deterministic shuffle of indices (seeded globally)
    idxs = list(range(n))
    random.shuffle(idxs)
    dev_idx = set(idxs[:n_dev])
    test_idx = set(idxs[n_dev:n_dev+n_test])
    train_idx = set(idxs[n_dev+n_test:])
    # preserve original within-group ordering for stability by sorting indices
    train = [group[i] for i in sorted(train_idx)]
    dev = [group[i] for i in sorted(dev_idx)]
    test = [group[i] for i in sorted(test_idx)]
    return train, dev, test

for pos in ['V','N','A','O']:
    g = all_by_pos[pos]
    tr, dv, ts = split_group(g)
    train_lines.extend(tr)
    dev_lines.extend(dv)
    test_lines.extend(ts)

# Final deterministic shuffle of each split to avoid POS-blocking from concatenation
random.shuffle(train_lines)
random.shuffle(dev_lines)
random.shuffle(test_lines)

# Verify POS ratios per split
from collections import Counter

def pos_counts(lines):
    c = Counter()
    for lemma, msd, form in lines:
        if msd.startswith('V'):
            c['V'] += 1
        elif msd.startswith('N'):
            c['N'] += 1
        elif msd.startswith('A'):
            c['A'] += 1
        else:
            c['O'] += 1
    return c

c_all = { k: len(all_by_pos[k]) for k in ['V','N','A','O'] }
c_tr, c_dv, c_ts = pos_counts(train_lines), pos_counts(dev_lines), pos_counts(test_lines)

print("Totals by POS:", c_all)
print("Train POS:", c_tr)
print("Dev   POS:", c_dv)
print("Test  POS:", c_ts)
print("Sizes (train/dev/test):", len(train_lines), len(dev_lines), len(test_lines))

# Write files next to output_file, using a different base to avoid clobbering earlier artifacts
base = 'dsb_all'
trn_path = f"{base}.trn"
dev_path = f"{base}.dev"
tst_path = f"{base}.tst"

with open(trn_path, 'w', encoding='utf8') as ftr:
    for lemma, msd, form in train_lines:
        ftr.write(f"{lemma}\t{msd}\t{form}\n")
with open(dev_path, 'w', encoding='utf8') as fdev:
    for lemma, msd, form in dev_lines:
        fdev.write(f"{lemma}\t{msd}\t{form}\n")
with open(tst_path, 'w', encoding='utf8') as ftst:
    for lemma, msd, form in test_lines:
        ftst.write(f"{lemma}\t{msd}\t{form}\n")

print("Wrote:")
print(" -", trn_path, len(train_lines))
print(" -", dev_path, len(dev_lines))
print(" -", tst_path, len(test_lines))

# Guarantee: all verbs are used (they are split among the three); verify no verb loss
verb_total = c_all['V']
verb_used = c_tr['V'] + c_dv['V'] + c_ts['V']
print(f"Verb usage check: total={verb_total}, used={verb_used}, ok={verb_total == verb_used}")

Totals by POS: {'V': 2898, 'N': 12372, 'A': 4851, 'O': 0}
Train POS: Counter({'N': 9898, 'A': 3881, 'V': 2318})
Dev   POS: Counter({'N': 1237, 'A': 485, 'V': 290})
Test  POS: Counter({'N': 1237, 'A': 485, 'V': 290})
Sizes (train/dev/test): 16097 2012 2012
Wrote:
 - dsb_all.trn 16097
 - dsb_all.dev 2012
 - dsb_all.tst 2012
Verb usage check: total=2898, used=2898, ok=True


# Step 3: Create fixed-size splits (10k/1k/1k) with minimal lemma overlap
Goal: deterministically build train/dev/test with sizes 10,000 / 1,000 / 1,000 from the verbs-only file `dsb` (created in Step 1), avoiding lemma overlap as much as possible. If exact sizes cannot be met without overlap, we only split lemmas across splits as a last resort to hit the exact counts.

In [9]:
# Step 3: POS-proportional, lemma-aware 10k/1k/1k splits with minimal lemma overlap
import os
from collections import defaultdict, Counter
import math
import random
# Deterministic mixing seed to avoid overly-sorted outputs
random.seed(42)

# Inputs/outputs
input_path = 'dsb'  # columns: lemma\tmsd\tform (written by Step 1)
base_out = 'dsb'  # will write base_out.trn/dev/tst

# Targets
TRAIN_TARGET, DEV_TARGET, TEST_TARGET = 10_000, 1_000, 1_000

splits = ['train', 'dev', 'test']
targets = {'train': TRAIN_TARGET, 'dev': DEV_TARGET, 'test': TEST_TARGET}

if not os.path.exists(input_path):
    print(f"Input file '{input_path}' not found. Run Step 1 or adjust 'input_path'.")
else:
    # Read and bucket by POS and lemma
    pos_lemmas = {'V': defaultdict(list), 'N': defaultdict(list), 'A': defaultdict(list), 'O': defaultdict(list)}
    pos_counts = Counter()
    total_rows = 0
    with open(input_path, 'r', encoding='utf8') as fin:
        for raw in fin:
            parts = raw.rstrip('\n').split('\t')
            if len(parts) != 3:
                continue
            lemma, msd, form = parts
            rec = (lemma, msd, form)
            if msd.startswith('V'):
                pos='V'
            elif msd.startswith('N'):
                pos='N'
            elif msd.startswith('A'):
                pos='A'
            else:
                pos='O'
            pos_lemmas[pos][lemma].append(rec)
            pos_counts[pos] += 1
            total_rows += 1

    desired_total = TRAIN_TARGET + DEV_TARGET + TEST_TARGET
    # If not enough rows, reduce targets deterministically (preserve dev/test first)
    if total_rows < desired_total:
        print(f"Warning: dataset has only {total_rows} rows < desired {desired_total}. Targets will be reduced.")
        # keep dev/test up to requested, reduce train first
        train_t = max(0, min(TRAIN_TARGET, total_rows - DEV_TARGET - TEST_TARGET))
        rem = total_rows - train_t
        dev_t = min(DEV_TARGET, rem // 2)
        test_t = min(TEST_TARGET, rem - dev_t)
        targets = {'train': train_t, 'dev': dev_t, 'test': test_t}
    else:
        targets = {'train': TRAIN_TARGET, 'dev': DEV_TARGET, 'test': TEST_TARGET}

    print('Total rows:', total_rows, 'POS counts:', dict(pos_counts))

    # Compute per-split, per-POS allocations proportional to POS frequencies
    # Use floor + largest-remainder method to ensure sums exactly match each split target
    def allocate_pos_for_split(split_target):
        # compute raw floats
        raw = {}
        for p in pos_lemmas.keys():
            raw[p] = (pos_counts[p] / total_rows) * split_target if total_rows > 0 else 0.0
        floor_alloc = {p: math.floor(v) for p, v in raw.items()}
        rem = split_target - sum(floor_alloc.values())
        # sort positions by fractional remainder desc
        fracs = sorted(pos_lemmas.keys(), key=lambda p: raw[p] - floor_alloc[p], reverse=True)
        alloc = dict(floor_alloc)
        i = 0
        while rem > 0 and i < len(fracs):
            alloc[fracs[i]] += 1
            rem -= 1
            i += 1
        return alloc

    pos_alloc = {s: allocate_pos_for_split(targets[s]) for s in splits}

    # Ensure every POS that exists in data appears at least once in each split if possible
    for s in splits:
        for p in pos_lemmas.keys():
            if pos_counts[p] > 0 and pos_alloc[s][p] == 0:
                # try to steal one from another POS in same split with alloc>1
                donor = next((q for q in pos_lemmas.keys() if pos_alloc[s][q] > 1), None)
                if donor is not None:
                    pos_alloc[s][donor] -= 1
                    pos_alloc[s][p] += 1

    print('Per-split POS allocation samples (train/dev/test):')
    for p in pos_lemmas.keys():
        print(p, [pos_alloc[s][p] for s in splits])

    # Now assign lemmas within each POS to splits trying to keep whole-lemma assignments
    assigned = {s: [] for s in splits}
    overlap_lemmas = set()

    for p in pos_lemmas.keys():
        lem_groups = pos_lemmas[p]  # lemma -> list of recs
        # build ordered lemmas by group size desc, shuffle within same size deterministically
        size_buckets = {}
        for lem in lem_groups.keys():
            size_buckets.setdefault(len(lem_groups[lem]), []).append(lem)
        ordered_lemmas = []
        for size in sorted(size_buckets.keys(), reverse=True):
            bucket = size_buckets[size]
            random.shuffle(bucket)
            ordered_lemmas.extend(bucket)

        # remaining quotas for this POS per split
        rem_pos = {s: pos_alloc[s][p] for s in splits}
        pending = []
        # First pass: assign whole lemmas where they fit
        for lem in ordered_lemmas:
            grp = lem_groups[lem]
            gsize = len(grp)
            candidates = [s for s in splits if rem_pos[s] >= gsize]
            if candidates:
                # choose split with largest remaining quota (stable tie-breaker: train>dev>test)
                best = max(candidates, key=lambda s: (rem_pos[s], 1 if s=='train' else 0 if s=='dev' else -1))
                assigned[best].extend(grp)
                rem_pos[best] -= gsize
            else:
                pending.append(lem)

        # Second pass: partially assign pending lemmas to fill remaining quotas
        if any(rem_pos[s] > 0 for s in splits) and pending:
            random.shuffle(pending)
            for lem in pending:
                grp = lem_groups[lem]
                idx = 0
                while sum(rem_pos.values()) > 0 and idx < len(grp):
                    # pick split with largest remaining for this POS
                    best = max(splits, key=lambda s: rem_pos[s])
                    if rem_pos[best] <= 0:
                        break
                    take = min(rem_pos[best], len(grp) - idx)
                    if take > 0:
                        assigned[best].extend(grp[idx:idx+take])
                        rem_pos[best] -= take
                        idx += take
                        overlap_lemmas.add(lem)
                    else:
                        break

        # If any small rounding differences remain, they will be fixed later by trimming

    # Final assembly: ensure each split has exact target size by deterministic trimming if needed
    for s in splits:
        random.shuffle(assigned[s])

    # Trim or report if short
    final = {}
    for s in splits:
        tgt = targets[s]
        if len(assigned[s]) >= tgt:
            final[s] = assigned[s][:tgt]
        else:
            # if short, we will fill from other splits' surplus deterministically (rare)
            final[s] = assigned[s][:]

    # If any split is short, fill from others deterministically
    for s in splits:
        if len(final[s]) < targets[s]:
            need = targets[s] - len(final[s])
            # gather candidates from other splits extras (beyond their target) or from their tail
            donors = [x for x in splits if x != s]
            for d in donors:
                while need > 0 and len(final[d]) > targets[d]:
                    # move from end to keep deterministic behavior
                    final[s].append(final[d].pop())
                    need -= 1
            # if still need, take from others' tails (deterministic pop)
            for d in donors:
                while need > 0 and len(final[d]) > 0:
                    final[s].append(final[d].pop())
                    need -= 1

    # Write outputs
    trn_path = f"{base_out}.trn"
    dev_path = f"{base_out}.dev"
    tst_path = f"{base_out}.tst"
    with open(trn_path, 'w', encoding='utf8') as ftr:
        for lemma, msd, form in final['train']:
            ftr.write(f"{lemma}\t{msd}\t{form}\n")
    with open(dev_path, 'w', encoding='utf8') as fdev:
        for lemma, msd, form in final['dev']:
            fdev.write(f"{lemma}\t{msd}\t{form}\n")
    with open(tst_path, 'w', encoding='utf8') as ftst:
        for lemma, msd, form in final['test']:
            ftst.write(f"{lemma}\t{msd}\t{form}\n")

    # Reporting
    def pos_counts_lines(lines):
        c = Counter()
        for lemma, msd, form in lines:
            if msd.startswith('V'): c['V'] += 1
            elif msd.startswith('N'): c['N'] += 1
            elif msd.startswith('A'): c['A'] += 1
            else: c['O'] += 1
        return c

    print('\nWrote outputs:')
    print(' -', trn_path, len(final['train']))
    print(' -', dev_path, len(final['dev']))
    print(' -', tst_path, len(final['test']))
    print('\nPOS counts per split:')
    print('Train:', pos_counts_lines(final['train']))
    print('Dev:  ', pos_counts_lines(final['dev']))
    print('Test: ', pos_counts_lines(final['test']))
    # Lemma overlap stats
    def lemma_set(lines): return {lem for (lem, _msd, _form) in lines}
    lt, ld, ls = map(lemma_set, (final['train'], final['dev'], final['test']))
    print('\nLemma overlaps (train∩dev, train∩test, dev∩test):', (len(lt & ld), len(lt & ls), len(ld & ls)))
    if overlap_lemmas:
        print('Lemmas forcibly split across splits to reach exact sizes:', len(overlap_lemmas))


Total rows: 20121 POS counts: {'N': 12372, 'A': 4851, 'V': 2898}
Per-split POS allocation samples (train/dev/test):
V [1440, 144, 144]
N [6149, 615, 615]
A [2411, 241, 241]
O [0, 0, 0]

Wrote outputs:
 - dsb.trn 10000
 - dsb.dev 1000
 - dsb.tst 1000

POS counts per split:
Train: Counter({'N': 6149, 'A': 2411, 'V': 1440})
Dev:   Counter({'N': 615, 'A': 241, 'V': 144})
Test:  Counter({'N': 615, 'A': 241, 'V': 144})

Lemma overlaps (train∩dev, train∩test, dev∩test): (2, 3, 3)
Lemmas forcibly split across splits to reach exact sizes: 5
